# Milestone 2 Lab, notebook 1 of 6: Where we are, and the business's records

## Where we are

| Notebook | What you do |
|---|---|
| **1. Where we are, and the business's records** (you are here) | Recall the question and the answer key, and explore the records profit is calculated from |
| 2. One calculation, two engines | Write a calculation by hand in pandas and in SQL, with test cases that judge both |
| 3. A model drafts a transformation | Ask a model to write the calculation, and let the tests judge it |
| 4. Which model? Accuracy, cost, and speed | Give the same tasks to several models and compare them |
| 5. An agent that tests and fixes its own code | Give a model the tests as a tool, and compare it with a single draft |
| 6. The pipeline, and the same question again | Run the calculations on each engine and ask the analyst the leader's question |

Model requests in this notebook: **0**. A full run of this lab costs well under a dollar.

## What this notebook is for

In milestone 1 you built an analyst that answers the leader's profit question and shows its working. One thing about it was not real: the profit figures were typed in by hand.

In this milestone the figures are **calculated from the business's own records**. This notebook introduces those records. There is no AI in it.

> If you have not done milestone 1, do it first. This lab assumes you know what an LLM call, a structured reply, a tool, and an agent are.

In [1]:
import pandas as pd
from lab_helpers import load_result, load_source_tables, show_doc

## Step 1: What the business is asking for now

Every milestone starts with a requirement. Here is this one's, from the **BRD**.

In [2]:
show_doc("BRD.md", "1. Business problem")

> **From `docs/BRD.md`: 1. Business problem**

In milestone 1, Kestrel Ridge Outfitters got an analyst that answers profit questions and shows its working. But the profit figures behind it were prepared by hand. Nobody can say how the trail water bottle's $5,188.20 was arrived at, and nobody could reproduce it next month.

Finance will not sign off on a number that cannot be recalculated. The business needs every profit figure to be **calculated from its own records**, by rules that are written down, the same way every time.

Two more needs come with that:

- **The calculation must outlive the tools it runs on.** Today the data is small. As the retailer grows, the same rules will have to run on much larger data, with different technology. The rules must not need rewriting when that happens.
- **AI should help write the calculations, and the business wants to know which AI to use.** Models differ in how well they do this work, how fast they are, and what they cost.

The leader's question has not changed, and neither has the right answer. This is the answer key you worked out by hand in milestone 1. By the end of this lab, calculations must produce these same figures from raw records.

In [3]:
answer_key = load_result("answer_key")

print("Confidence:", answer_key["confidence"], " Data problems:", answer_key["flags"])
pd.DataFrame(answer_key["figures"])[["what", "entity_id", "kind", "value"]]

Confidence: low  Data problems: ['QF-001', 'QF-002']


,what,entity_id,kind,value
0,#1 product: Trail water bottle,P001,value,5188.20
1,#2 product: Headlamp,P008,value,4250.00
2,#3 product: Rain jacket,P009,value,3841.00
3,Top category: Hydration,Hydration,value,13358.06
4,"Hydration, change since the month before",Hydration,change,376.06
5,"Apparel, change since the month before (the bi...",Apparel,change,-5914.87
6,"Camp, change since the month before (cannot be...",Camp,change,NaN
7,Camp stove (its cost is missing),P007,value,NaN


## Step 2: Meet the business's records

On the business workflow page you watched a product travel from vendor to platform to customer, and saw the records each step leaves behind. These are those records: seven tables.

In [4]:
tables = load_source_tables()

pd.DataFrame([{"table": name, "rows": len(frame), "columns": ", ".join(frame.columns)} for name, frame in tables.items()])

,table,rows,columns
0,products,12,"product_id, product_name, category"
1,orders,2450,"order_id, order_date, status"
2,order_items,2545,"order_item_id, order_id, product_id, quantity,..."
3,refunds,49,"refund_id, order_item_id, product_id, refund_d..."
4,shipments,2521,"shipment_id, order_item_id, product_id, ship_d..."
5,payment_fees,2521,"fee_id, order_item_id, product_id, capture_dat..."
6,product_costs,23,"product_id, month, unit_cost"


An **order** is one purchase. An **order item** is one product on that order (a "line"). Most orders have one line; some have two.

In [5]:
tables["orders"].head(3)

,order_id,order_date,status
0,ORD-20001,2026-09-01,completed
1,ORD-20002,2026-09-01,completed
2,ORD-20003,2026-09-01,completed


In [6]:
tables["order_items"].head(3)

,order_item_id,order_id,product_id,quantity,unit_list_price,discount_amount
0,OI-00001,ORD-20001,P001,1,40.0,2.0
1,OI-00002,ORD-20002,P001,1,40.0,2.0
2,OI-00003,ORD-20003,P001,1,40.0,2.0


After a line is ordered, three more things can happen to it, and each leaves a record in its own table: it **ships** (which costs money to pick, pack, and carry), the payment provider takes a **fee**, and sometimes the customer gets a **refund**.

## Step 3: Follow one order line by hand

Let's take one line and find everything the business knows about it. This is the profit calculation in miniature.

In [7]:
line = tables["order_items"].iloc[0]
line_id = line["order_item_id"]

print("The line:")
display(tables["order_items"][tables["order_items"]["order_item_id"] == line_id])
print("Its order:")
display(tables["orders"][tables["orders"]["order_id"] == line["order_id"]])
print("Its shipment:")
display(tables["shipments"][tables["shipments"]["order_item_id"] == line_id])
print("Its payment fee:")
display(tables["payment_fees"][tables["payment_fees"]["order_item_id"] == line_id])

The line:


,order_item_id,order_id,product_id,quantity,unit_list_price,discount_amount
0,OI-00001,ORD-20001,P001,1,40.0,2.0


Its order:


,order_id,order_date,status
0,ORD-20001,2026-09-01,completed


Its shipment:


,shipment_id,order_item_id,product_id,ship_date,pick_pack_cost,carrier_cost
0,SHP-00001,OI-00001,P001,2026-09-04,2.5,4.0


Its payment fee:


,fee_id,order_item_id,product_id,capture_date,fee_amount
0,FEE-00001,OI-00001,P001,2026-09-01,1.14


Now work out what this one line earned. Read each number off the tables above.

In [8]:
shipment = tables["shipments"][tables["shipments"]["order_item_id"] == line_id].iloc[0]
fee = tables["payment_fees"][tables["payment_fees"]["order_item_id"] == line_id].iloc[0]
month = tables["orders"][tables["orders"]["order_id"] == line["order_id"]].iloc[0]["order_date"].strftime("%Y-%m")
costs = tables["product_costs"]
unit_cost = costs[(costs["product_id"] == line["product_id"]) & (costs["month"] == month)].iloc[0]["unit_cost"]

gross_sales = line["quantity"] * line["unit_list_price"]
net_sales = gross_sales - line["discount_amount"]
gross_profit = net_sales - line["quantity"] * unit_cost
contribution_profit = gross_profit - shipment["pick_pack_cost"] - shipment["carrier_cost"] - fee["fee_amount"]

print(f"Gross sales          {gross_sales:8.2f}   quantity x list price")
print(f"Net sales            {net_sales:8.2f}   less the discount (this line has no refund)")
print(f"Gross profit         {gross_profit:8.2f}   less quantity x unit cost")
print(f"Contribution profit  {contribution_profit:8.2f}   less pick and pack, carrier, and payment fee")

Gross sales             40.00   quantity x list price
Net sales               38.00   less the discount (this line has no refund)
Gross profit            20.00   less quantity x unit cost
Contribution profit     12.36   less pick and pack, carrier, and payment fee


That is all a profit calculation is: the same sum, done for every line, and added up by product and month. There are about 2,500 lines, so we need code to do it. Writing that code is this milestone.

## Step 4: The records are not tidy

Real records have problems. Milestone 1 listed three as "quality flags". Now you can see them in the records themselves.

**A missing cost.** The camp stove (P007) has a cost for September and none for October.

In [9]:
tables["product_costs"][tables["product_costs"]["product_id"] == "P007"]

,product_id,month,unit_cost
12,P007,2026-09,44.0


**Duplicated lines.** Three trail water bottle lines appear twice. The same line was loaded twice by mistake.

In [10]:
items = tables["order_items"]
items[items.duplicated(keep=False)]

,order_item_id,order_id,product_id,quantity,unit_list_price,discount_amount
1245,OI-01246,ORD-21207,P001,1,40.0,4.0
1246,OI-01246,ORD-21207,P001,1,40.0,4.0
1750,OI-01750,ORD-21695,P001,1,40.0,4.0
1751,OI-01750,ORD-21695,P001,1,40.0,4.0
1925,OI-01924,ORD-21860,P001,1,40.0,4.0
1926,OI-01924,ORD-21860,P001,1,40.0,4.0


**A refund bigger than the sale.** A sun hat line worth $25.00 has a $60.00 refund.

In [11]:
refunds = tables["refunds"]
refunds[(refunds["product_id"] == "P011") & (refunds["refund_amount"] > 25)].merge(items.drop_duplicates(), on=["order_item_id", "product_id"])[["refund_id", "order_item_id", "refund_date", "refund_amount", "quantity", "unit_list_price"]]

,refund_id,order_item_id,refund_date,refund_amount,quantity,unit_list_price
0,REF-0002,OI-00080,2026-09-07,60.0,1,25.0


Two more things are in there on purpose. They are not problems, just facts a calculation must handle correctly.

**Cancelled orders.** They have lines, but they are not sales.

In [12]:
print(tables["orders"]["status"].value_counts())

status
completed    2426
cancelled      24
Name: count, dtype: int64


**Orders refunded in a later month.** These were ordered in September and refunded in October. Which month should the refund count in?

In [13]:
refunded = refunds.merge(items.drop_duplicates()[["order_item_id", "order_id"]], on="order_item_id").merge(tables["orders"], on="order_id")
refunded[refunded["order_date"].dt.month != refunded["refund_date"].dt.month][["refund_id", "product_id", "order_date", "refund_date", "refund_amount"]]

,refund_id,product_id,order_date,refund_date,refund_amount
27,REF-0028,P004,2026-09-26,2026-10-03,52.25
28,REF-0029,P010,2026-09-29,2026-10-03,16.20
30,REF-0031,P012,2026-09-20,2026-10-05,63.75
31,REF-0032,P001,2026-09-26,2026-10-05,36.00


## Step 5: The rules

Questions like "which month does a refund count in?" are business decisions, not coding decisions. The BRD records them.

In [14]:
show_doc("BRD.md", "6. Business rules agreed with the learner")

> **From `docs/BRD.md`: 6. Business rules agreed with the learner**

The rules from milestone 1 still apply. These are added or made exact.

| Rule | Decision |
|---|---|
| Same figures as milestone 1 | The business records are generated so that the calculated figures equal milestone 1's prepared figures exactly |
| When a sale counts | In the month the order was placed |
| When a refund counts | In the month the refund was issued, even if the order was placed in an earlier month |
| When delivery costs count | Pick-and-pack and carrier costs count in the month the item shipped |
| When payment fees count | In the month the payment was captured |
| Cancelled orders | Not a sale. Nothing from a cancelled order is counted |
| Missing product cost | The product's cost, gross profit, and contribution profit are unavailable for that month. Never zero, never estimated |
| Duplicated records | Counted as they appear. Finding and removing them is the job of the quality checks in milestone 3. Until then the known duplicates stay flagged |
| Which engines | The calculations run on two engines now: pandas (for small data) and SQL (for medium data). The design must leave room for PySpark (for large data) |
| Which models | Only models available through OpenRouter. Nothing is installed on the learner's computer |
| Who decides if AI's work is accepted | Tests with hand-worked examples decide. A person reviews anything that goes into production |

Notice the rule about duplicated records: **counted as they appear**. Finding and removing duplicates is the job of the quality checks in milestone 3. If we quietly removed them now, the figures would stop matching milestone 1's, and we would have changed two things at once.

## What we now know

- Profit figures must now be **calculated** from seven tables of business records.
- The calculation for one line is simple. The work is doing it for every line, correctly, by clear rules.
- The records contain a missing cost, duplicated lines, an oversized refund, cancelled orders, and refunds that cross months. A correct calculation has to deal with each of them.
- The answer key from milestone 1 is our target. The calculated figures must equal it.

**Next: notebook 2**, where you write the first calculation by hand, twice.